In [10]:
from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Sabida"
STUDENT_ID = "TUPM-26-1157"
SEED_DIGIT = 7
ID_SUM = 22

BASE_DIR = Path("Lab6_File_Handling_Additional_Exercises")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
BACKUP_DIR = BASE_DIR / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("Directories ready:", BASE_DIR.resolve())

Directories ready: C:\Users\Mojo\Documents\ComputerProgram\Lab6_FileHandling_Sabida\Lab6_File_Handling_Additional_Exercises


In [11]:
log_file = DATA_DIR / "weblog.csv"
records_limit = 1000 + ID_SUM             
target_class = (SEED_DIGIT % 4) + 2        
target_prefix = str(target_class)

filtered_csv_1 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex1_filtered_weblog.csv"
json_summary_1 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex1_weblog_summary.json"
backup_json_1 = BACKUP_DIR / json_summary_1.name

records_processed = 0
matched_rows = []
status_counter = Counter()
url_counter = Counter()

with open(log_file, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
    reader = csv.DictReader(infile)
    for row in reader:
        if records_processed >= records_limit:
            break
        records_processed += 1

        ip = (row.get("IP") or "").strip()
        timestamp = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        # Dataset uses "Staus" as the column header
        status = (row.get("Staus") or row.get("Status") or "").strip()

        if not ip or not url or not status.isdigit():
            continue

        if status.startswith(target_prefix):
            matched_rows.append({
                "IP": ip,
                "Time": timestamp,
                "URL": url,
                "Staus": status
            })
            status_counter[status] += 1
            url_counter[url] += 1

# Export filtered CSV
with open(filtered_csv_1, mode="w", encoding="utf-8", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=["IP", "Time", "URL", "Staus"])
    writer.writeheader()
    writer.writerows(matched_rows)

# Build and write JSON summary
top_5_urls = [{"url": u, "count": c} for u, c in url_counter.most_common(5)]
summary_1 = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "target_status_class": f"{target_class}xx",
    "records_in_target_class": len(matched_rows),
    "count_per_status_code": dict(sorted(status_counter.items())),
    "five_most_requested_urls": top_5_urls,
    "filtered_csv_file": filtered_csv_1.name
}

with open(json_summary_1, mode="w", encoding="utf-8") as jfile:
    json.dump(summary_1, jfile, indent=4)

# Create and verify JSON backup
shutil.copy2(json_summary_1, backup_json_1)
backup_verified_1 = backup_json_1.exists() and backup_json_1.stat().st_size == json_summary_1.stat().st_size

print("=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===")
print("Records processed:", records_processed)
print("Target status class:", f"{target_class}xx")
print("Records in target class:", len(matched_rows))
print("Count per status code:", dict(sorted(status_counter.items())))
print("Five most requested URLs:", top_5_urls)
print("Filtered CSV file:", filtered_csv_1)
print("JSON summary file:", json_summary_1)
print("Backup verification:", backup_verified_1)

=== EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS ===
Records processed: 1022
Target status class: 5xx
Records in target class: 0
Count per status code: {}
Five most requested URLs: []
Filtered CSV file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex1_filtered_weblog.csv
JSON summary file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex1_weblog_summary.json
Backup verification: True


In [12]:
mobile_file = DATA_DIR / "user_behavior_dataset.csv"
target_os = "Android" if SEED_DIGIT % 2 == 0 else "iOS"   # iOS
target_behavior_class = (SEED_DIGIT % 5) + 1              # 3

exported_fields = [
    "User ID",
    "Device Model",
    "Operating System",
    "Battery Drain (mAh/day)",
    "Data Usage (MB/day)"
]

filtered_csv_2 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex2_mobile_profile.csv"
json_summary_2 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex2_mobile_summary.json"
backup_csv_2 = BACKUP_DIR / filtered_csv_2.name

total_records = 0
os_matched_count = 0
selected_rows = []
total_battery = 0.0
total_data_usage = 0.0

with open(mobile_file, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
    reader = csv.DictReader(infile)
    for row in reader:
        total_records += 1
        os_val = (row.get("Operating System") or "").strip()
        if os_val.lower() != target_os.lower():
            continue
        os_matched_count += 1

        try:
            b_class = int(float((row.get("User Behavior Class") or "").strip()))
            battery = float((row.get("Battery Drain (mAh/day)") or "").strip())
            data_mb = float((row.get("Data Usage (MB/day)") or "").strip())
        except ValueError:
            continue

        if b_class == target_behavior_class:
            selected_rows.append({field: (row.get(field) or "").strip() for field in exported_fields})
            total_battery += battery
            total_data_usage += data_mb

selected_count = len(selected_rows)
avg_battery = round(total_battery / selected_count, 2) if selected_count > 0 else 0.0
avg_data = round(total_data_usage / selected_count, 2) if selected_count > 0 else 0.0

# Export filtered CSV
with open(filtered_csv_2, mode="w", encoding="utf-8", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=exported_fields)
    writer.writeheader()
    writer.writerows(selected_rows)

# Verify exported CSV row count against selected_count
with open(filtered_csv_2, mode="r", encoding="utf-8", newline="") as checkfile:
    exported_row_count = sum(1 for _ in csv.DictReader(checkfile))

count_verified = (exported_row_count == selected_count)

# Write JSON summary
summary_2 = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "operating_system": target_os,
    "user_behavior_class": target_behavior_class,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matched_count,
    "records_in_target_behavior_class": selected_count,
    "average_battery_drain_mAh_per_day": avg_battery,
    "average_data_usage_MB_per_day": avg_data,
    "exported_row_count": exported_row_count,
    "count_verification": count_verified
}

with open(json_summary_2, mode="w", encoding="utf-8") as jfile:
    json.dump(summary_2, jfile, indent=4)

# Create CSV backup and check size
shutil.copy2(filtered_csv_2, backup_csv_2)
backup_size_bytes = backup_csv_2.stat().st_size if backup_csv_2.exists() else 0

print("=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===")
print(f"Operating system and behavior class: {target_os}, Class {target_behavior_class}")
print("Total records in dataset:", total_records)
print("Records matching operating system:", os_matched_count)
print("Records in target behavior class:", selected_count)
print("Average battery drain (mAh/day):", avg_battery)
print("Average data usage (MB/day):", avg_data)
print("Count verification:", count_verified)
print("Exported CSV file:", filtered_csv_2)
print("JSON summary file:", json_summary_2)
print(f"Backup file and size: {backup_csv_2} ({backup_size_bytes} bytes)")

=== EXERCISE 2: MOBILE DEVICE USAGE PROFILING ===
Operating system and behavior class: iOS, Class 3
Total records in dataset: 700
Records matching operating system: 146
Records in target behavior class: 32
Average battery drain (mAh/day): 1492.03
Average data usage (MB/day): 828.09
Count verification: True
Exported CSV file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex2_mobile_profile.csv
JSON summary file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex2_mobile_summary.json
Backup file and size: Lab6_File_Handling_Additional_Exercises\backup\Sabida_TUPM-26-1157_ex2_mobile_profile.csv (973 bytes)


In [14]:
cyber_file = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
target_year = 2015 + SEED_DIGIT   # 2022

filtered_csv_3 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_cyber_2022.csv"
json_summary_3 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_top5_costliest.json"
backup_json_3 = BACKUP_DIR / json_summary_3.name
report_txt_3 = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_ex3_incident_report.txt"

total_records_3 = 0
year_incidents = []
attack_counter = Counter()
total_loss = 0.0
valid_loss_count = 0
csv_headers = []

with open(cyber_file, mode="r", encoding="utf-8", errors="replace", newline="") as infile:
    reader = csv.DictReader(infile)
    csv_headers = reader.fieldnames or []

    for idx, row in enumerate(reader):
        total_records_3 += 1
        year_str = (row.get("Year") or "").strip()
        if not year_str.isdigit() or int(year_str) != target_year:
            continue

        attack_type = (row.get("Attack Type") or "Unknown").strip()
        attack_counter[attack_type] += 1

        loss_raw = (row.get("Financial Loss (in Million $)") or "").strip()
        try:
            loss_val = float(loss_raw)
            total_loss += loss_val
            valid_loss_count += 1
        except ValueError:
            loss_val = None

        year_incidents.append((idx, loss_val, row))

incident_count = len(year_incidents)
avg_loss = round(total_loss / valid_loss_count, 3) if valid_loss_count > 0 else 0.0
total_loss_rounded = round(total_loss, 3)

# Sort descending by loss; retain original dataset order (idx ascending) on ties
ranked_incidents = sorted(
    [item for item in year_incidents if item[1] is not None],
    key=lambda x: (-x[1], x[0])
)
top_5_records = [item[2] for item in ranked_incidents[:5]]

# Export filtered CSV of target-year incidents
with open(filtered_csv_3, mode="w", encoding="utf-8", newline="") as outfile:
    writer = csv.DictWriter(outfile, fieldnames=csv_headers)
    writer.writeheader()
    for _, _, row in year_incidents:
        writer.writerow(row)

# Export top 5 costliest incidents to JSON
json_payload_3 = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "target_year": target_year,
    "incidents_in_target_year": incident_count,
    "count_per_attack_type": dict(attack_counter),
    "total_financial_loss_million_usd": total_loss_rounded,
    "average_financial_loss_million_usd": avg_loss,
    "five_costliest_incidents": top_5_records
}

with open(json_summary_3, mode="w", encoding="utf-8") as jfile:
    json.dump(json_payload_3, jfile, indent=4)

# Create and verify backup of JSON file
shutil.copy2(json_summary_3, backup_json_3)
backup_verified_3 = backup_json_3.exists() and backup_json_3.stat().st_size == json_summary_3.stat().st_size

# Generate text report
with open(report_txt_3, mode="w", encoding="utf-8") as rfile:
    rfile.write("CYBERSECURITY INCIDENT REPORT\n")
    rfile.write("--------------------------------------------------\n")
    rfile.write(f"Student: {LAST_NAME} ({STUDENT_ID})\n")
    rfile.write(f"Target Year: {target_year}\n")
    rfile.write(f"Total Records in Dataset: {total_records_3}\n")
    rfile.write(f"Incidents in Target Year: {incident_count}\n\n")
    rfile.write("Incident Count per Attack Type:\n")
    for attack, count in attack_counter.items():
        rfile.write(f"  - {attack}: {count}\n")
    rfile.write(f"\nTotal Financial Loss (USD Millions): {total_loss_rounded}\n")
    rfile.write(f"Average Financial Loss (USD Millions): {avg_loss}\n\n")
    rfile.write("Five Costliest Incidents:\n")
    for rank, inc in enumerate(top_5_records, start=1):
        rfile.write(
            f"  {rank}. {inc.get('Country', 'N/A')} | {inc.get('Attack Type', 'N/A')} | "
            f"${inc.get('Financial Loss (in Million $)', 'N/A')}M\n"
        )
    rfile.write("\nOutput Files:\n")
    rfile.write(f"  - Filtered CSV: {filtered_csv_3.name}\n")
    rfile.write(f"  - JSON Summary: {json_summary_3.name}\n")
    rfile.write(f"  - JSON Backup Verified: {backup_verified_3}\n")

print("=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===")
print("Target year:", target_year)
print("Total records in dataset:", total_records_3)
print("Incidents in target year:", incident_count)
print("Count per attack type:", dict(attack_counter))
print("Total financial loss (USD millions):", total_loss_rounded)
print("Average financial loss (USD millions):", avg_loss)
print("Five costliest incidents:", len(top_5_records), "exported to JSON")
print("Filtered CSV file:", filtered_csv_3)
print("JSON summary file:", json_summary_3)
print("Backup verification:", backup_verified_3)
print("Report file:", report_txt_3)

=== EXERCISE 3: CYBERSECURITY INCIDENT REPORT ===
Target year: 2022
Total records in dataset: 3000
Incidents in target year: 318
Count per attack type: {'Phishing': 57, 'Malware': 64, 'Ransomware': 56, 'SQL Injection': 46, 'DDoS': 54, 'Man-in-the-Middle': 41}
Total financial loss (USD millions): 15870.86
Average financial loss (USD millions): 49.908
Five costliest incidents: 5 exported to JSON
Filtered CSV file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex3_cyber_2022.csv
JSON summary file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex3_top5_costliest.json
Backup verification: True
Report file: Lab6_File_Handling_Additional_Exercises\output\Sabida_TUPM-26-1157_ex3_incident_report.txt
